<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 04
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Feature Engineering & Dataset Assembly
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Consolidates cleaned price action, fetches external market indicators, and generates cross-sectional technical features for global machine learning models.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Completed Steps in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Load Processed Datasets:</b> Consolidate asset classes, handle forward-fills, and align start dates.</li>
            <li><b>Pre-Engineering Data Quality Verification:</b> Executes a lightweight EDA on raw adjusted close series to verify basic integrity, check for zero/negative prices, flag extreme price moves (&gt;50%), and validate start-date alignments.</li>
            <li><b>Fetch Yahoo Features:</b> Retrieve OHLCV, dividends, and stock splits for tradable assets.</li>
            <li><b>Merge Market Data:</b> Unify market data with synthetic assets into a single master schema.</li>
            <li><b>Technical Feature Engineering:</b> Generate vectorized returns, rolling volatility, drawdown, and volume features.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Feature EDA & Redundancy Check &rarr; Forward Target Engineering (t+20) &rarr; Walk-Forward Model Training
    </p>
</div>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from pathlib import Path
sys.path.append(os.path.abspath(".."))

from src.data_loader import build_master_dataset
from src.fetch_features import fetch_yfinance_features
from src.feature_engineering import prepare_market_data, prepare_master_market_data

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Load Processed Datasets
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Loads all asset classes (i.e only Adjused Close), strips weekends and holidays using the master stock trading calendar, merges into a unified DataFrame, and aligns to the common start date.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Note: Uses datasets previously generated in the <code>03_risk_analysis.ipynb</code> pipeline.
        </p>
    </div>
</div>

In [ ]:
master_df = build_master_dataset()

DATA_DIR =Path("../data/features")
output_path = Path(DATA_DIR) / ("verified_asset_data.csv")
master_df.to_csv(output_path)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Sanity Check
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Pre-Engineering Data Quality Verification
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Executes a lightweight EDA on the raw adjusted close series before feature engineering. Verifies basic data integrity, checks for zero/negative prices, flags extreme price moves (>50%), and validates start-date alignments across all assets.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Note: Ensuring structural soundness here prevents mathematical errors (e.g., in np.log) from propagating into downstream technical features.
        </p>
    </div>
</div>

In [ ]:
# 1. FIX: Ensure Date is the index for proper time-series vector math
if 'Date' in master_df.columns:
    master_df.set_index('Date', inplace=True)
    
# 2. Filter out non-price columns safely
price_cols = [col for col in master_df.select_dtypes(include='number').columns 
              if 'return' not in col.lower() and 'vol' not in col.lower()]
price_df = master_df[price_cols]

print(f"--- ANALYZING {len(price_cols)} PRICE COLUMNS ---\n")

# --- CHECK 1: ZERO OR NEGATIVE PRICES ---
print("--- 1. ZERO OR NEGATIVE PRICE CHECK ---")
invalid_prices = (price_df <= 0).sum()
if invalid_prices.sum() > 0:
    print("⚠️ WARNING: Found zero or negative prices in the following assets:")
    print(invalid_prices[invalid_prices > 0])
else:
    print("✅ PASS: No zero or negative prices found.\n")

# --- CHECK 2: EXTREME SINGLE-DAY MOVES (>50%) ---
print("--- 2. EXTREME SINGLE-DAY MOVE CHECK (>50%) ---")
daily_returns = price_df.pct_change()
extreme_mask = (daily_returns > 0.50) | (daily_returns < -0.50)
extreme_instances = extreme_mask.stack()
extreme_instances = extreme_instances[extreme_instances].index

if not extreme_instances.empty:
    print(f"🚨 WARNING: Found {len(extreme_instances)} extreme single-day moves in price series.")
    print("Note: Plausible for Crypto, but highly suspicious for Bonds/Commodities/Stocks/ETFs/RealEstate.")
    
    prev_prices = price_df.shift(1)
    report_data = []
    
    for date_idx, ticker in extreme_instances:
        ret = daily_returns.loc[date_idx, ticker]
        prev_price = prev_prices.loc[date_idx, ticker]
        curr_price = price_df.loc[date_idx, ticker]
        
        report_data.append({
            'Date': date_idx,
            'Ticker': ticker,
            'Daily Return (%)': round(ret * 100, 2),
            'Price Before': prev_price,
            'Price After': curr_price
        })

    extreme_df = pd.DataFrame(report_data)
    if pd.api.types.is_datetime64_any_dtype(extreme_df['Date']):
        extreme_df['Date'] = extreme_df['Date'].dt.strftime('%Y-%m-%d')
        
    display(extreme_df.sort_values(by='Date', ascending=False))
else:
    print("✅ PASS: No extreme price moves (>50%) found.\n")

# --- CHECK 3: START DATE ALIGNMENT ---
print("\n--- 3. START DATE ALIGNMENT CHECK ---")
history_stats = pd.DataFrame({
    'Start Date': price_df.apply(lambda x: x.first_valid_index()),
    'End Date': price_df.apply(lambda x: x.last_valid_index()),
    'Total Trading Days': price_df.notna().sum()
})

print("Date Boundaries per Asset (Sorted by Start Date):")
display(history_stats.sort_values('Start Date', ascending=False))

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Observation
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Extreme Moves Validated
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        For the tickers flagged with a daily return above the 50% threshold, the dates and prices were checked manually in Yahoo Finance. It is confirmed that this accurately reflects the Adjusted Close price data recorded for those specific days (reflecting true market events rather than unadjusted split artifacts).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Action: No raw data patching is required. Proceeding safely to feature engineering.
        </p>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Fetch Yahoo Finance Features
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Downloads incremental market data for all asset classes (Stocks, ETFs, Crypto Currency, Commodities) handles currency conversion to EUR, fetches and appends up-to-date synthetic ETF series, and aligns all records onto a master stock business calendar while cleaning corporate action fields as per their asset class.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>yfinance_raw_features.parquet</code>
        </p>
    </div>
</div>

In [ ]:
yf_features = fetch_yfinance_features()
yf_features.head()

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Merge Market Data
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Merges the Adjusted Close data (Bond, and Real estate) with Yahoo Finance features into one unified master dataset ready for feature computation.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>merged_market_dataset.parquet</code>
        </p>
    </div>
</div>

In [ ]:
if master_df.index.name == 'Date':
    master_df.reset_index(inplace=True)
market_df = prepare_market_data(master_df, yf_features)
market_df.head()

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Technical Feature Engineering (Vectorized)
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Computes grouped cross-sectional indicators: <b>Log Returns</b> (1, 5, 20, 60D), <b>Rolling Volatility</b>, <b>SMA/EMA Ratios</b>, <b>MACD</b>, <b>RSI-14</b>, <b>Bollinger Width</b>, zero-warmup <b>Max Drawdown (60D)</b>, and <b>OBV Z-Scores</b>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>master_dataset.parquet</code>
        </p>
    </div>
</div>

In [ ]:
feature_df = prepare_master_market_data(market_df)
feature_df.head()